# Flux Variability Analysis (FVA) on TCA Cycle Model

This notebook loads the `TCA_cycle` model, applies specific flux constraints with a 10% range, and performs FVA.

In [12]:
import cobra
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import flux_variability_analysis
import pandas as pd
import os

In [13]:
# Load the model
model_path = "../../models/TCA_cycle.json"
model = cobra.io.load_json_model(model_path)
print(f"Loaded model '{model.id}' with {len(model.reactions)} reactions and {len(model.metabolites)} metabolites.")

Loaded model 'metabolic_network' with 32 reactions and 22 metabolites.


## Apply Flux Constraints

We fix the following fluxes with a range of +/- 10%:
- R16 (PYR_B): 0.0700
- R17 (AKG_B): 0.2300
- R18 (OAA_B): 0.1200
- R19 (VALX): 0.0500
- R20 (LYSX): 0.0300

In [34]:
# Define target fluxes and calculate ranges
targets = {
    'R16': 70,
    'R17': 230,
    'R18': 120,
    'R19': 50,
    'R20': 30
}

print("Applying constraints (Target +/- 10%):")
for rxn_id, value in targets.items():
    delta = value * 0.1
    lb = value - delta
    ub = value + delta
    
    # Update model bounds
    model.reactions.get_by_id(rxn_id).bounds = (lb, ub)
    print(f"{rxn_id}: Target={value:.4f}, Range=[{lb:.4f}, {ub:.4f}]")


model.reactions.get_by_id("R1").bounds = (1000, 1000)



Applying constraints (Target +/- 10%):
R16: Target=70.0000, Range=[63.0000, 77.0000]
R17: Target=230.0000, Range=[207.0000, 253.0000]
R18: Target=120.0000, Range=[108.0000, 132.0000]
R19: Target=50.0000, Range=[45.0000, 55.0000]
R20: Target=30.0000, Range=[27.0000, 33.0000]


## Perform Flux Variability Analysis

In [35]:
pd.set_option('display.max_rows', None)
# Run FVA
# fraction_of_optimum=0.0 allows exploring the full feasible space given constraints
fva_result = flux_variability_analysis(model, fraction_of_optimum=0.0, processes=1)
print("\nFVA Results:")
print(fva_result)


FVA Results:
               minimum      maximum
R1         1000.000000  1000.000000
R2            0.000000   330.142857
R3          153.714286   351.800000
R4          153.714286   351.800000
R5          153.714286   351.800000
R6            0.000000   230.857143
R7            0.000000  1000.000000
R8            0.000000  1000.000000
R9            0.000000   571.800000
R10         480.181818  1000.000000
R11         230.666667   450.600000
R12           0.000000   115.428571
R13         230.666667   500.000000
R14           0.000000   462.200000
R15           0.000000  1000.000000
R16          63.000000    77.000000
R17         207.000000   253.000000
R18         108.000000   132.000000
R19          45.000000    55.000000
R20          27.000000    33.000000
R21          10.200000   562.428571
EX_PYR_EX -1000.000000 -1000.000000
EX_GLU_EX  -330.142857     0.000000
EX_O2_EX   -500.000000  -230.666667
EX_CO2_EX     0.000000   462.200000
DM_PYR_B     63.000000    77.000000
DM_AKG_B    20

## Generate 30 Flux Combinations for R2, R8, R9

In [ ]:
import numpy as np
from scipy.stats.qmc import LatinHypercube

# FVA ranges for R2, R8, R9
fva_ranges = {
    'R2': (fva_result.loc['R2', 'minimum'], fva_result.loc['R2', 'maximum']),
    'R8': (fva_result.loc['R8', 'minimum'], fva_result.loc['R8', 'maximum']),
    'R9': (fva_result.loc['R9', 'minimum'], fva_result.loc['R9', 'maximum']),
}

print("FVA ranges used for sampling:")
for rxn, (lb, ub) in fva_ranges.items():
    print(f"  {rxn}: [{lb:.4f}, {ub:.4f}]")

# Latin Hypercube Sampling: 30 combinations across 3 dimensions
sampler = LatinHypercube(d=3, seed=42)
lhs_samples = sampler.random(n=30)  # shape (30, 3), values in [0, 1]

# Scale to FVA bounds
bounds = list(fva_ranges.values())
scaled = np.column_stack([
    lhs_samples[:, i] * (ub - lb) + lb
    for i, (lb, ub) in enumerate(bounds)
])

flux_combinations = pd.DataFrame(scaled, columns=['R2', 'R8', 'R9'])
flux_combinations.index = range(1, 31)
flux_combinations.index.name = 'Sample'

pd.set_option('display.float_format', '{:.4f}'.format)
print(f"\n30 flux combinations (LHS) for R2, R8, R9:")
print(flux_combinations)
